# RescueTwin AI - Creación del dataset integrado

Este notebook tiene como objetivo construir el dataset final del proyecto **RescueTwin AI**, un gemelo digital de un robot cuadrúpedo para rescate en zonas de derrumbe.

El dataset final integrará información de:

- Sensores ambientales.
- Sensores de gases.
- Estado operativo de batería.
- Variables simuladas del terreno.
- Detección simulada de personas.
- Nivel de riesgo operativo.
- Acción recomendada para el robot.

El archivo final generado será:

`data/processed/rescuetwin_dataset.csv`

In [42]:
import os
import pandas as pd
import numpy as np

np.random.seed(42)

In [43]:
RAW_PATH = "../data/raw"
PROCESSED_PATH = "../data/processed"

INDOOR_PATH = "../data/raw/indoor_environment"
GAS_PATH = "../data/raw/gas_sensor_uci"
BATTERY_PATH = "../data/raw/battery_kaggle_cleaned"
SARD_PATH = "../data/raw/sard"

os.makedirs(PROCESSED_PATH, exist_ok=True)

In [44]:
carpetas_esperadas = [
    INDOOR_PATH,
    GAS_PATH,
    BATTERY_PATH,
    SARD_PATH,
    PROCESSED_PATH
]

for carpeta in carpetas_esperadas:
    if os.path.exists(carpeta):
        print(f"OK - Existe: {carpeta}")
    else:
        print(f"ERROR - No existe: {carpeta}")

OK - Existe: ../data/raw/indoor_environment
OK - Existe: ../data/raw/gas_sensor_uci
OK - Existe: ../data/raw/battery_kaggle_cleaned
OK - Existe: ../data/raw/sard
OK - Existe: ../data/processed


In [45]:
archivos_datos = []

extensiones_validas = (".csv", ".txt", ".dat", ".data")

for root, dirs, files in os.walk(RAW_PATH):
    for file in files:
        if file.lower().endswith(extensiones_validas):
            ruta = os.path.join(root, file)
            archivos_datos.append(ruta)

print("Cantidad total de archivos de datos encontrados:", len(archivos_datos))

print("\nPrimeros 30 archivos encontrados:")
for archivo in archivos_datos[:30]:
    print(archivo)

Cantidad total de archivos de datos encontrados: 13343

Primeros 30 archivos encontrados:
../data/raw/battery_kaggle_cleaned/metadata.csv
../data/raw/battery_kaggle_cleaned/extra_infos/README_45_46_47_48.txt
../data/raw/battery_kaggle_cleaned/extra_infos/README_53_54_55_56.txt
../data/raw/battery_kaggle_cleaned/extra_infos/README_49_50_51_52.txt
../data/raw/battery_kaggle_cleaned/extra_infos/README_38_39_40.txt
../data/raw/battery_kaggle_cleaned/extra_infos/README_33_34_36.txt
../data/raw/battery_kaggle_cleaned/extra_infos/README_29_30_31_32.txt
../data/raw/battery_kaggle_cleaned/extra_infos/README_05_06_07_18.txt
../data/raw/battery_kaggle_cleaned/extra_infos/README_25_26_27_28.txt
../data/raw/battery_kaggle_cleaned/extra_infos/README_41_42_43_44.txt
../data/raw/battery_kaggle_cleaned/data/06825.csv
../data/raw/battery_kaggle_cleaned/data/01192.csv
../data/raw/battery_kaggle_cleaned/data/03785.csv
../data/raw/battery_kaggle_cleaned/data/04954.csv
../data/raw/battery_kaggle_cleaned/dat

In [46]:
indoor_files = [f for f in archivos_datos if "indoor_environment" in f]
gas_files = [f for f in archivos_datos if "gas_sensor_uci" in f]
battery_files = [f for f in archivos_datos if "battery_kaggle_cleaned" in f]
sard_files = [f for f in archivos_datos if "sard" in f]

print("Cantidad de archivos por fuente:")
print("Indoor Environment:", len(indoor_files))
print("Gas Sensor UCI:", len(gas_files))
print("Battery Kaggle Cleaned:", len(battery_files))
print("SARD:", len(sard_files))

Cantidad de archivos por fuente:
Indoor Environment: 1
Gas Sensor UCI: 10
Battery Kaggle Cleaned: 7575
SARD: 5757


In [47]:
print("Archivos Indoor:")
for f in indoor_files[:10]:
    print(f)

print("\nArchivos Gas:")
for f in gas_files[:20]:
    print(f)

print("\nArchivo de metadata de batería:")
battery_metadata = [f for f in battery_files if "metadata.csv" in f]

if len(battery_metadata) > 0:
    print(battery_metadata[0])
else:
    print("No se encontró metadata.csv")

Archivos Indoor:
../data/raw/indoor_environment/rpi_23_reg_with_IAQ.csv

Archivos Gas:
../data/raw/gas_sensor_uci/batch8.dat
../data/raw/gas_sensor_uci/batch9.dat
../data/raw/gas_sensor_uci/batch4.dat
../data/raw/gas_sensor_uci/batch5.dat
../data/raw/gas_sensor_uci/batch7.dat
../data/raw/gas_sensor_uci/batch6.dat
../data/raw/gas_sensor_uci/batch2.dat
../data/raw/gas_sensor_uci/batch3.dat
../data/raw/gas_sensor_uci/batch1.dat
../data/raw/gas_sensor_uci/batch10.dat

Archivo de metadata de batería:
../data/raw/battery_kaggle_cleaned/metadata.csv


In [48]:
def leer_archivo_seguro(ruta):
    """
    Intenta leer archivos de datos con distintos separadores.
    Soporta CSV, TXT, DAT y DATA.
    """
    intentos = [
        {"sep": ","},
        {"sep": ";"},
        {"sep": "\t"},
        {"sep": " "},
        {"sep": r"\s+", "engine": "python"}
    ]
    
    for params in intentos:
        try:
            df = pd.read_csv(ruta, **params)
            if df.shape[1] > 1:
                return df
        except:
            pass
    
    print(f"No se pudo leer correctamente: {ruta}")
    return None

In [49]:
archivos_clave = []

if len(indoor_files) > 0:
    archivos_clave.append(indoor_files[0])

if len(gas_files) > 0:
    archivos_clave.append(gas_files[0])

if len(battery_metadata) > 0:
    archivos_clave.append(battery_metadata[0])

for archivo in archivos_clave:
    print("\n==============================")
    print("Archivo:", archivo)
    
    df_temp = leer_archivo_seguro(archivo)
    
    if df_temp is not None:
        print("Filas y columnas:", df_temp.shape)
        print("Columnas:")
        print(df_temp.columns.tolist())
        display(df_temp.head())


Archivo: ../data/raw/indoor_environment/rpi_23_reg_with_IAQ.csv
Filas y columnas: (280030, 13)
Columnas:
['id', 'date_time', 'rpi_id', 'proximity', 'humidity', 'pressure', 'light', 'temperature', 'sound_high', 'sound_mid', 'sound_low', 'sound_amp', 'IAQ_score']


,id,date_time,rpi_id,proximity,humidity,pressure,light,temperature,sound_high,sound_mid,sound_low,sound_amp,IAQ_score
0,2021-06-16 19:26:27_23,2021-06-16 19:26:27,23,0,20.569237,677.633021,214.96985,21.723234,15.976453,25.115489,80.460706,20.258775,38.236481
1,2021-06-16 19:27:28_23,2021-06-16 19:27:28,23,0,20.164035,944.208435,212.08965,22.445803,16.219735,33.374927,78.124848,21.286585,38.249233
2,2021-06-16 19:28:31_23,2021-06-16 19:28:31,23,0,19.927352,944.182027,209.76240,22.464315,29.031231,33.226572,75.257702,22.919251,38.277590
3,2021-06-16 19:29:33_23,2021-06-16 19:29:33,23,0,19.524638,944.240214,205.99505,22.948009,23.976011,51.710558,88.640952,27.387920,38.818883
4,2021-06-16 19:30:35_23,2021-06-16 19:30:35,23,0,19.516710,677.633021,203.66780,22.832800,74.785816,47.173401,72.817866,32.462847,39.564203



Archivo: ../data/raw/gas_sensor_uci/batch8.dat
Filas y columnas: (293, 2)
Columnas:
['4', '100.000000 1:191.678400 2:1.097676 3:0.085713 4:0.374642 5:2.811447 6:-0.119149 7:-0.365198 8:-2.850503 9:37699.609300 10:3.410369 11:9.692600 12:14.618374 13:24.715278 14:-8.165545 15:-12.094239 16:-34.006684 17:8932.682900 18:3.424292 19:2.519343 20:4.476731 21:9.334217 22:-2.016334 23:-3.410085 24:-9.919937 25:9036.281500 26:3.303704 27:2.540368 28:4.411801 29:7.623647 30:-2.004905 31:-3.284269 32:-10.582721 33:2397.743200 34:2.298949 35:0.858986 36:2.086802 37:5.056685 38:-0.588877 39:-1.144501 40:-5.434928 41:2149.170800 42:2.187579 43:0.772232 44:2.044545 45:4.349598 46:-0.529639 47:-1.112689 48:-4.452800 49:11864.302200 50:3.680024 51:3.746102 52:7.045698 53:10.688661 54:-2.856566 55:-5.107787 56:-13.541820 57:12503.934100 58:3.631380 59:3.994928 60:7.655898 61:11.861445 62:-3.033432 63:-5.399332 64:-12.707092 65:33209.031200 66:3.313124 67:6.974960 68:9.710674 69:16.653879 70:-5.851903 7

,4,100.000000 1:191.678400 2:1.097676 3:0.085713 4:0.374642 5:2.811447 6:-0.119149 7:-0.365198 8:-2.850503 9:37699.609300 10:3.410369 11:9.692600 12:14.618374 13:24.715278 14:-8.165545 15:-12.094239 16:-34.006684 17:8932.682900 18:3.424292 19:2.519343 20:4.476731 21:9.334217 22:-2.016334 23:-3.410085 24:-9.919937 25:9036.281500 26:3.303704 27:2.540368 28:4.411801 29:7.623647 30:-2.004905 31:-3.284269 32:-10.582721 33:2397.743200 34:2.298949 35:0.858986 36:2.086802 37:5.056685 38:-0.588877 39:-1.144501 40:-5.434928 41:2149.170800 42:2.187579 43:0.772232 44:2.044545 45:4.349598 46:-0.529639 47:-1.112689 48:-4.452800 49:11864.302200 50:3.680024 51:3.746102 52:7.045698 53:10.688661 54:-2.856566 55:-5.107787 56:-13.541820 57:12503.934100 58:3.631380 59:3.994928 60:7.655898 61:11.861445 62:-3.033432 63:-5.399332 64:-12.707092 65:33209.031200 66:3.313124 67:6.974960 68:9.710674 69:16.653879 70:-5.851903 71:-8.429603 72:-26.192714 73:21114.495100 74:2.735633 75:4.193540 76:5.835873 77:10.278766 78:-3.500111 79:-5.037374 80:-15.469851 81:10877.833900 82:3.608550 83:2.834880 84:4.323547 85:6.233371 86:-2.287414 87:-3.360956 88:-7.785282 89:8926.402300 90:3.514866 91:2.361454 92:3.708356 93:6.265744 94:-1.906495 95:-2.824825 96:-6.552052 97:3281.951100 98:2.352164 99:1.139184 100:2.904699 101:4.703665 102:-0.761658 103:-1.397183 104:-3.552804 105:3224.382300 106:2.335492 107:1.120283 108:2.875418 109:4.791037 110:-0.742220 111:-1.282544 112:-4.057572 113:14282.364800 114:3.973612 115:4.081411 116:6.915375 117:9.600034 118:-3.273612 119:-5.293610 120:-10.223869 121:12723.271000 122:3.888402 123:3.740372 124:6.324860 125:8.530802 126:-2.909987 127:-4.585015 128:-8.863957
0,5,25.000000 1:1026.771100 2:1.778150 3:0.217603 ...
1,4,100.000000 1:311.874800 2:1.195577 3:0.105731 ...
2,5,25.000000 1:987.016900 2:1.843827 3:0.218607 4...
3,4,100.000000 1:152.406300 2:1.100063 3:0.076945 ...
4,5,25.000000 1:938.891600 2:1.853794 3:0.200317 4...



Archivo: ../data/raw/battery_kaggle_cleaned/metadata.csv
Filas y columnas: (7565, 10)
Columnas:
['type', 'start_time', 'ambient_temperature', 'battery_id', 'test_id', 'uid', 'filename', 'Capacity', 'Re', 'Rct']


,type,start_time,ambient_temperature,battery_id,test_id,uid,filename,Capacity,Re,Rct
0,discharge,[2010. 7. 21. 15. 0. ...,4,B0047,0,1,00001.csv,1.6743047446975208,NaN,NaN
1,impedance,[2010. 7. 21. 16. 53. ...,24,B0047,1,2,00002.csv,NaN,0.05605783343888099,0.20097016584458333
2,charge,[2010. 7. 21. 17. 25. ...,4,B0047,2,3,00003.csv,NaN,NaN,NaN
3,impedance,[2010 7 21 20 31 5],24,B0047,3,4,00004.csv,NaN,0.05319185850921101,0.16473399914864734
4,discharge,[2.0100e+03 7.0000e+00 2.1000e+01 2.1000e+01 2...,4,B0047,4,5,00005.csv,1.5243662105099023,NaN,NaN


In [50]:
# Tomamos el primer CSV encontrado dentro de indoor_environment.
# Si hubiera más de uno, después se puede cambiar manualmente por el más útil.

if len(indoor_files) == 0:
    raise FileNotFoundError("No se encontraron archivos CSV en indoor_environment")

indoor_file = indoor_files[0]
df_indoor = leer_archivo_seguro(indoor_file)

print("Archivo ambiental usado:", indoor_file)
print("Dimensiones:", df_indoor.shape)
print("Columnas:")
print(df_indoor.columns.tolist())

display(df_indoor.head())

Archivo ambiental usado: ../data/raw/indoor_environment/rpi_23_reg_with_IAQ.csv
Dimensiones: (280030, 13)
Columnas:
['id', 'date_time', 'rpi_id', 'proximity', 'humidity', 'pressure', 'light', 'temperature', 'sound_high', 'sound_mid', 'sound_low', 'sound_amp', 'IAQ_score']


,id,date_time,rpi_id,proximity,humidity,pressure,light,temperature,sound_high,sound_mid,sound_low,sound_amp,IAQ_score
0,2021-06-16 19:26:27_23,2021-06-16 19:26:27,23,0,20.569237,677.633021,214.96985,21.723234,15.976453,25.115489,80.460706,20.258775,38.236481
1,2021-06-16 19:27:28_23,2021-06-16 19:27:28,23,0,20.164035,944.208435,212.08965,22.445803,16.219735,33.374927,78.124848,21.286585,38.249233
2,2021-06-16 19:28:31_23,2021-06-16 19:28:31,23,0,19.927352,944.182027,209.76240,22.464315,29.031231,33.226572,75.257702,22.919251,38.277590
3,2021-06-16 19:29:33_23,2021-06-16 19:29:33,23,0,19.524638,944.240214,205.99505,22.948009,23.976011,51.710558,88.640952,27.387920,38.818883
4,2021-06-16 19:30:35_23,2021-06-16 19:30:35,23,0,19.516710,677.633021,203.66780,22.832800,74.785816,47.173401,72.817866,32.462847,39.564203


In [51]:
df_indoor.columns = (
    df_indoor.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

print(df_indoor.columns.tolist())

['id', 'date_time', 'rpi_id', 'proximity', 'humidity', 'pressure', 'light', 'temperature', 'sound_high', 'sound_mid', 'sound_low', 'sound_amp', 'iaq_score']


In [52]:
def buscar_columna(df, posibles_nombres):
    """
    Busca una columna en el dataframe a partir de una lista de posibles nombres.
    Devuelve el nombre real de la columna si existe.
    """
    columnas = df.columns.tolist()
    
    for posible in posibles_nombres:
        for col in columnas:
            if posible.lower() in col.lower():
                return col
    
    return None

In [53]:
N = 2000

ambiental = pd.DataFrame()

col_temp = buscar_columna(df_indoor, ["temperature", "temperatura", "temp"])
col_hum = buscar_columna(df_indoor, ["humidity", "humedad", "hum"])
col_pressure = buscar_columna(df_indoor, ["pressure", "presion", "presión"])
col_light = buscar_columna(df_indoor, ["light", "luz", "illumination"])
col_sound = buscar_columna(df_indoor, ["sound", "noise", "sonido", "ruido"])
col_co2 = buscar_columna(df_indoor, ["co2", "carbon"])
col_pm25 = buscar_columna(df_indoor, ["pm2.5", "pm25", "particulate", "particles"])

print("Columnas detectadas:")
print("Temperatura:", col_temp)
print("Humedad:", col_hum)
print("Presión:", col_pressure)
print("Luz:", col_light)
print("Sonido:", col_sound)
print("CO2:", col_co2)
print("PM25:", col_pm25)

Columnas detectadas:
Temperatura: temperature
Humedad: humidity
Presión: pressure
Luz: light
Sonido: sound_high
CO2: None
PM25: None


In [54]:
# Temperatura
if col_temp:
    ambiental["temperatura"] = pd.to_numeric(
        df_indoor[col_temp].sample(N, replace=True).values,
        errors="coerce"
    )
else:
    ambiental["temperatura"] = np.random.normal(28, 6, N)

# Humedad
if col_hum:
    ambiental["humedad"] = pd.to_numeric(
        df_indoor[col_hum].sample(N, replace=True).values,
        errors="coerce"
    )
else:
    ambiental["humedad"] = np.random.normal(55, 15, N)

# Presión
if col_pressure:
    ambiental["presion"] = pd.to_numeric(
        df_indoor[col_pressure].sample(N, replace=True).values,
        errors="coerce"
    )
else:
    ambiental["presion"] = np.random.normal(1013, 15, N)

# Luz
if col_light:
    ambiental["luz"] = pd.to_numeric(
        df_indoor[col_light].sample(N, replace=True).values,
        errors="coerce"
    )
else:
    ambiental["luz"] = np.random.uniform(0, 100, N)

# Sonido
if col_sound:
    ambiental["sonido_db"] = pd.to_numeric(
        df_indoor[col_sound].sample(N, replace=True).values,
        errors="coerce"
    )
else:
    ambiental["sonido_db"] = np.random.uniform(25, 95, N)

# CO2
if col_co2:
    ambiental["co2"] = pd.to_numeric(
        df_indoor[col_co2].sample(N, replace=True).values,
        errors="coerce"
    )
else:
    ambiental["co2"] = np.random.uniform(400, 2500, N)

# Partículas
if col_pm25:
    ambiental["particulas_pm25"] = pd.to_numeric(
        df_indoor[col_pm25].sample(N, replace=True).values,
        errors="coerce"
    )
else:
    ambiental["particulas_pm25"] = np.random.uniform(5, 180, N)

ambiental.head()

,temperatura,humedad,presion,luz,sonido_db,co2,particulas_pm25
0,23.742562,37.156921,951.246360,0.59255,54.215643,505.075257,136.682603
1,22.381415,37.422549,677.633021,0.00000,70.790609,545.730161,168.990470
2,22.333412,19.738998,941.234135,0.00000,50.419725,2085.626242,20.303590
3,21.993103,40.284164,938.066586,0.53330,18.377416,2098.968669,102.632636
4,24.145010,42.892223,944.360592,0.47405,34.733886,915.139906,49.429148


In [55]:
ambiental = ambiental.fillna(ambiental.median(numeric_only=True))

ambiental["temperatura"] = ambiental["temperatura"].clip(0, 60)
ambiental["humedad"] = ambiental["humedad"].clip(0, 100)
ambiental["presion"] = ambiental["presion"].clip(900, 1100)
ambiental["luz"] = ambiental["luz"].clip(0, 100)
ambiental["sonido_db"] = ambiental["sonido_db"].clip(0, 120)
ambiental["co2"] = ambiental["co2"].clip(300, 5000)
ambiental["particulas_pm25"] = ambiental["particulas_pm25"].clip(0, 300)

ambiental.describe()

,temperatura,humedad,presion,luz,sonido_db,co2,particulas_pm25
count,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000
mean,22.066611,33.262511,931.324131,1.197809,33.679281,1435.710787,93.113170
std,2.044869,7.926508,22.219832,5.321829,17.794980,608.041380,50.060173
min,13.632610,13.848169,900.000000,0.000000,3.399427,400.388910,5.020763
25%,20.643200,26.628829,900.000000,0.000000,20.426184,930.772529,50.526273
50%,22.182346,34.755090,943.522421,0.000000,29.728107,1429.161414,93.813686
75%,23.674634,38.904536,948.546536,0.533300,42.672035,1947.521880,136.500762
max,28.218289,50.293425,960.055426,100.000000,76.791244,2499.234813,179.840884


In [56]:
if len(gas_files) == 0:
    print("No se encontraron archivos de gases en gas_sensor_uci.")
    print("Se van a simular variables de gas para poder continuar el proyecto.")
    df_gas = None
else:
    gas_file = gas_files[0]
    df_gas = leer_archivo_seguro(gas_file)

    print("Archivo de gases usado:", gas_file)

    if df_gas is not None:
        print("Dimensiones:", df_gas.shape)
        print("Columnas:")
        print(df_gas.columns.tolist())
        display(df_gas.head())
    else:
        print("No se pudo leer el archivo de gases. Se van a simular variables de gas.")

Archivo de gases usado: ../data/raw/gas_sensor_uci/batch8.dat
Dimensiones: (293, 2)
Columnas:
['4', '100.000000 1:191.678400 2:1.097676 3:0.085713 4:0.374642 5:2.811447 6:-0.119149 7:-0.365198 8:-2.850503 9:37699.609300 10:3.410369 11:9.692600 12:14.618374 13:24.715278 14:-8.165545 15:-12.094239 16:-34.006684 17:8932.682900 18:3.424292 19:2.519343 20:4.476731 21:9.334217 22:-2.016334 23:-3.410085 24:-9.919937 25:9036.281500 26:3.303704 27:2.540368 28:4.411801 29:7.623647 30:-2.004905 31:-3.284269 32:-10.582721 33:2397.743200 34:2.298949 35:0.858986 36:2.086802 37:5.056685 38:-0.588877 39:-1.144501 40:-5.434928 41:2149.170800 42:2.187579 43:0.772232 44:2.044545 45:4.349598 46:-0.529639 47:-1.112689 48:-4.452800 49:11864.302200 50:3.680024 51:3.746102 52:7.045698 53:10.688661 54:-2.856566 55:-5.107787 56:-13.541820 57:12503.934100 58:3.631380 59:3.994928 60:7.655898 61:11.861445 62:-3.033432 63:-5.399332 64:-12.707092 65:33209.031200 66:3.313124 67:6.974960 68:9.710674 69:16.653879 70:-5

,4,100.000000 1:191.678400 2:1.097676 3:0.085713 4:0.374642 5:2.811447 6:-0.119149 7:-0.365198 8:-2.850503 9:37699.609300 10:3.410369 11:9.692600 12:14.618374 13:24.715278 14:-8.165545 15:-12.094239 16:-34.006684 17:8932.682900 18:3.424292 19:2.519343 20:4.476731 21:9.334217 22:-2.016334 23:-3.410085 24:-9.919937 25:9036.281500 26:3.303704 27:2.540368 28:4.411801 29:7.623647 30:-2.004905 31:-3.284269 32:-10.582721 33:2397.743200 34:2.298949 35:0.858986 36:2.086802 37:5.056685 38:-0.588877 39:-1.144501 40:-5.434928 41:2149.170800 42:2.187579 43:0.772232 44:2.044545 45:4.349598 46:-0.529639 47:-1.112689 48:-4.452800 49:11864.302200 50:3.680024 51:3.746102 52:7.045698 53:10.688661 54:-2.856566 55:-5.107787 56:-13.541820 57:12503.934100 58:3.631380 59:3.994928 60:7.655898 61:11.861445 62:-3.033432 63:-5.399332 64:-12.707092 65:33209.031200 66:3.313124 67:6.974960 68:9.710674 69:16.653879 70:-5.851903 71:-8.429603 72:-26.192714 73:21114.495100 74:2.735633 75:4.193540 76:5.835873 77:10.278766 78:-3.500111 79:-5.037374 80:-15.469851 81:10877.833900 82:3.608550 83:2.834880 84:4.323547 85:6.233371 86:-2.287414 87:-3.360956 88:-7.785282 89:8926.402300 90:3.514866 91:2.361454 92:3.708356 93:6.265744 94:-1.906495 95:-2.824825 96:-6.552052 97:3281.951100 98:2.352164 99:1.139184 100:2.904699 101:4.703665 102:-0.761658 103:-1.397183 104:-3.552804 105:3224.382300 106:2.335492 107:1.120283 108:2.875418 109:4.791037 110:-0.742220 111:-1.282544 112:-4.057572 113:14282.364800 114:3.973612 115:4.081411 116:6.915375 117:9.600034 118:-3.273612 119:-5.293610 120:-10.223869 121:12723.271000 122:3.888402 123:3.740372 124:6.324860 125:8.530802 126:-2.909987 127:-4.585015 128:-8.863957
0,5,25.000000 1:1026.771100 2:1.778150 3:0.217603 ...
1,4,100.000000 1:311.874800 2:1.195577 3:0.105731 ...
2,5,25.000000 1:987.016900 2:1.843827 3:0.218607 4...
3,4,100.000000 1:152.406300 2:1.100063 3:0.076945 ...
4,5,25.000000 1:938.891600 2:1.853794 3:0.200317 4...


In [57]:
if df_gas is not None:
    df_gas.columns = (
        df_gas.columns
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("-", "_")
    )

    print(df_gas.columns.tolist())
else:
    print("No hay dataset de gases cargado. Se usarán datos simulados.")

['4', '100.000000_1:191.678400_2:1.097676_3:0.085713_4:0.374642_5:2.811447_6:_0.119149_7:_0.365198_8:_2.850503_9:37699.609300_10:3.410369_11:9.692600_12:14.618374_13:24.715278_14:_8.165545_15:_12.094239_16:_34.006684_17:8932.682900_18:3.424292_19:2.519343_20:4.476731_21:9.334217_22:_2.016334_23:_3.410085_24:_9.919937_25:9036.281500_26:3.303704_27:2.540368_28:4.411801_29:7.623647_30:_2.004905_31:_3.284269_32:_10.582721_33:2397.743200_34:2.298949_35:0.858986_36:2.086802_37:5.056685_38:_0.588877_39:_1.144501_40:_5.434928_41:2149.170800_42:2.187579_43:0.772232_44:2.044545_45:4.349598_46:_0.529639_47:_1.112689_48:_4.452800_49:11864.302200_50:3.680024_51:3.746102_52:7.045698_53:10.688661_54:_2.856566_55:_5.107787_56:_13.541820_57:12503.934100_58:3.631380_59:3.994928_60:7.655898_61:11.861445_62:_3.033432_63:_5.399332_64:_12.707092_65:33209.031200_66:3.313124_67:6.974960_68:9.710674_69:16.653879_70:_5.851903_71:_8.429603_72:_26.192714_73:21114.495100_74:2.735633_75:4.193540_76:5.835873_77:10.2

In [58]:
gas = pd.DataFrame()

if df_gas is not None:
    col_concentration = buscar_columna(
        df_gas, 
        ["concentration", "concentracion", "concentración", "ppm"]
    )
    col_gas_type = buscar_columna(
        df_gas, 
        ["gas", "type", "class"]
    )

    print("Columna concentración detectada:", col_concentration)
    print("Columna tipo de gas detectada:", col_gas_type)

    if col_concentration:
        gas["gas_ppm"] = pd.to_numeric(
            df_gas[col_concentration].sample(N, replace=True).values,
            errors="coerce"
        )
    else:
        gas["gas_ppm"] = np.random.uniform(0, 500, N)

    if col_gas_type:
        gas["gas_tipo"] = df_gas[col_gas_type].sample(N, replace=True).values
    else:
        gas["gas_tipo"] = np.random.choice(
            ["sin_gas", "metano", "monoxido_carbono", "amoniaco", "humo", "gas_desconocido"],
            size=N,
            p=[0.35, 0.15, 0.20, 0.10, 0.15, 0.05]
        )

else:
    gas["gas_ppm"] = np.random.uniform(0, 500, N)
    gas["gas_tipo"] = np.random.choice(
        ["sin_gas", "metano", "monoxido_carbono", "amoniaco", "humo", "gas_desconocido"],
        size=N,
        p=[0.35, 0.15, 0.20, 0.10, 0.15, 0.05]
    )

gas["gas_ppm"] = pd.to_numeric(gas["gas_ppm"], errors="coerce")
gas["gas_ppm"] = gas["gas_ppm"].fillna(gas["gas_ppm"].median())
gas["gas_ppm"] = gas["gas_ppm"].clip(0, 500)

gas.head()

Columna concentración detectada: None
Columna tipo de gas detectada: None


,gas_ppm,gas_tipo
0,16.081237,gas_desconocido
1,447.921945,humo
2,326.041341,sin_gas
3,368.769094,amoniaco
4,1.303732,sin_gas


In [59]:
if len(battery_metadata) == 0:
    raise FileNotFoundError("No se encontró metadata.csv en battery_kaggle_cleaned")

battery_file = battery_metadata[0]
df_battery = leer_archivo_seguro(battery_file)

print("Archivo de batería usado:", battery_file)
print("Dimensiones:", df_battery.shape)
print("Columnas:")
print(df_battery.columns.tolist())

display(df_battery.head())

Archivo de batería usado: ../data/raw/battery_kaggle_cleaned/metadata.csv
Dimensiones: (7565, 10)
Columnas:
['type', 'start_time', 'ambient_temperature', 'battery_id', 'test_id', 'uid', 'filename', 'Capacity', 'Re', 'Rct']


,type,start_time,ambient_temperature,battery_id,test_id,uid,filename,Capacity,Re,Rct
0,discharge,[2010. 7. 21. 15. 0. ...,4,B0047,0,1,00001.csv,1.6743047446975208,NaN,NaN
1,impedance,[2010. 7. 21. 16. 53. ...,24,B0047,1,2,00002.csv,NaN,0.05605783343888099,0.20097016584458333
2,charge,[2010. 7. 21. 17. 25. ...,4,B0047,2,3,00003.csv,NaN,NaN,NaN
3,impedance,[2010 7 21 20 31 5],24,B0047,3,4,00004.csv,NaN,0.05319185850921101,0.16473399914864734
4,discharge,[2.0100e+03 7.0000e+00 2.1000e+01 2.1000e+01 2...,4,B0047,4,5,00005.csv,1.5243662105099023,NaN,NaN


In [60]:
df_battery.columns = (
    df_battery.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

print(df_battery.columns.tolist())

['type', 'start_time', 'ambient_temperature', 'battery_id', 'test_id', 'uid', 'filename', 'capacity', 're', 'rct']


In [61]:
battery = pd.DataFrame()

col_capacity = buscar_columna(df_battery, ["capacity"])
col_temp_battery = buscar_columna(df_battery, ["ambient_temperature", "temperature", "temperatura"])

print("Columna capacidad detectada:", col_capacity)
print("Columna temperatura detectada:", col_temp_battery)

# ==========================
# Batería restante
# ==========================

if col_capacity:
    capacidad = pd.Series(
        pd.to_numeric(
            df_battery[col_capacity].sample(N, replace=True).values,
            errors="coerce"
        )
    )
    
    capacidad = capacidad.fillna(capacidad.median())
    
    capacidad_min = capacidad.min()
    capacidad_max = capacidad.max()
    
    # Evita división por cero si todos los valores fueran iguales
    if capacidad_max != capacidad_min:
        battery["bateria"] = 100 * (capacidad - capacidad_min) / (capacidad_max - capacidad_min)
        battery["bateria"] = battery["bateria"].clip(15, 100)
    else:
        battery["bateria"] = np.random.randint(15, 100, N)
else:
    battery["bateria"] = np.random.randint(15, 100, N)

# ==========================
# Temperatura de batería
# ==========================

if col_temp_battery:
    temperatura_bateria = pd.Series(
        pd.to_numeric(
            df_battery[col_temp_battery].sample(N, replace=True).values,
            errors="coerce"
        )
    )
    
    temperatura_bateria = temperatura_bateria.fillna(temperatura_bateria.median())
    battery["temperatura_bateria"] = temperatura_bateria
else:
    battery["temperatura_bateria"] = np.random.uniform(25, 55, N)

battery["temperatura_bateria"] = battery["temperatura_bateria"].clip(0, 70)

# ==========================
# Voltaje de batería
# ==========================

battery["voltaje_bateria"] = np.round(
    10.5 + (battery["bateria"] / 100) * 2.1 + np.random.normal(0, 0.15, N),
    2
)

battery["voltaje_bateria"] = battery["voltaje_bateria"].clip(10.5, 12.6)

# ==========================
# Autonomía estimada
# ==========================

battery["autonomia_estimada_min"] = np.round(
    battery["bateria"] * 0.8 - battery["temperatura_bateria"] * 0.15,
    1
)

battery["autonomia_estimada_min"] = battery["autonomia_estimada_min"].clip(lower=0)

battery.head()

Columna capacidad detectada: capacity
Columna temperatura detectada: ambient_temperature


,bateria,temperatura_bateria,voltaje_bateria,autonomia_estimada_min
0,36.395081,4,10.82,28.5
1,61.886486,24,12.10,45.9
2,51.792509,24,11.46,37.8
3,60.061447,4,11.91,47.4
4,33.434138,24,11.06,23.1


In [62]:
df = pd.concat(
    [
        ambiental.reset_index(drop=True),
        gas.reset_index(drop=True),
        battery.reset_index(drop=True)
    ],
    axis=1
)

df.head()

,temperatura,humedad,presion,luz,sonido_db,co2,particulas_pm25,gas_ppm,gas_tipo,bateria,temperatura_bateria,voltaje_bateria,autonomia_estimada_min
0,23.742562,37.156921,951.246360,0.59255,54.215643,505.075257,136.682603,16.081237,gas_desconocido,36.395081,4,10.82,28.5
1,22.381415,37.422549,900.000000,0.00000,70.790609,545.730161,168.990470,447.921945,humo,61.886486,24,12.10,45.9
2,22.333412,19.738998,941.234135,0.00000,50.419725,2085.626242,20.303590,326.041341,sin_gas,51.792509,24,11.46,37.8
3,21.993103,40.284164,938.066586,0.53330,18.377416,2098.968669,102.632636,368.769094,amoniaco,60.061447,4,11.91,47.4
4,24.145010,42.892223,944.360592,0.47405,34.733886,915.139906,49.429148,1.303732,sin_gas,33.434138,24,11.06,23.1


In [63]:
df["vibracion"] = np.round(np.random.uniform(0, 2.5, N), 2)
df["inclinacion"] = np.round(np.random.uniform(0, 35, N), 1)
df["distancia_obstaculo"] = np.round(np.random.uniform(0.2, 5.0, N), 2)

df["velocidad_robot"] = np.round(np.random.uniform(0.1, 1.5, N), 2)
df["senal_comunicacion"] = np.round(np.random.uniform(20, 100, N), 1)

df.head()

,temperatura,humedad,presion,luz,sonido_db,co2,particulas_pm25,gas_ppm,gas_tipo,bateria,temperatura_bateria,voltaje_bateria,autonomia_estimada_min,vibracion,inclinacion,distancia_obstaculo,velocidad_robot,senal_comunicacion
0,23.742562,37.156921,951.246360,0.59255,54.215643,505.075257,136.682603,16.081237,gas_desconocido,36.395081,4,10.82,28.5,1.31,19.3,1.55,0.67,51.8
1,22.381415,37.422549,900.000000,0.00000,70.790609,545.730161,168.990470,447.921945,humo,61.886486,24,12.10,45.9,1.63,0.1,3.10,0.43,58.6
2,22.333412,19.738998,941.234135,0.00000,50.419725,2085.626242,20.303590,326.041341,sin_gas,51.792509,24,11.46,37.8,0.43,22.3,3.62,0.14,33.5
3,21.993103,40.284164,938.066586,0.53330,18.377416,2098.968669,102.632636,368.769094,amoniaco,60.061447,4,11.91,47.4,0.93,26.2,4.17,1.29,23.5
4,24.145010,42.892223,944.360592,0.47405,34.733886,915.139906,49.429148,1.303732,sin_gas,33.434138,24,11.06,23.1,2.33,8.7,4.65,0.56,35.1


In [64]:
df["visibilidad"] = np.round(
    100 
    - (df["particulas_pm25"] * 0.25)
    - (df["gas_ppm"] * 0.05)
    - np.random.uniform(0, 20, N),
    1
)

df["visibilidad"] = df["visibilidad"].clip(0, 100)

df[["particulas_pm25", "gas_ppm", "visibilidad"]].head()

,particulas_pm25,gas_ppm,visibilidad
0,136.682603,16.081237,45.2
1,168.990470,447.921945,28.5
2,20.303590,326.041341,72.8
3,102.632636,368.769094,54.9
4,49.429148,1.303732,74.9


In [65]:
df["persona_detectada"] = np.random.choice(
    [0, 1],
    size=N,
    p=[0.82, 0.18]
)

df["confianza_persona"] = np.where(
    df["persona_detectada"] == 1,
    np.random.uniform(0.60, 0.98, N),
    np.random.uniform(0.01, 0.40, N)
)

df["confianza_persona"] = np.round(df["confianza_persona"], 2)

df[["persona_detectada", "confianza_persona"]].head()

,persona_detectada,confianza_persona
0,0,0.11
1,0,0.05
2,0,0.14
3,0,0.12
4,0,0.19


In [66]:
zonas = [
    "Entrada",
    "Pasillo A",
    "Pasillo B",
    "Escalera colapsada",
    "Sector norte",
    "Sector sur",
    "Zona de escombros",
    "Habitación bloqueada",
    "Túnel estrecho",
    "Punto crítico"
]

df["zona"] = np.random.choice(zonas, size=N)

df[["zona"]].head()

,zona
0,Punto crítico
1,Punto crítico
2,Túnel estrecho
3,Túnel estrecho
4,Habitación bloqueada


In [67]:
def calcular_riesgo(row):
    puntaje = 0
    
    # Temperatura ambiental
    if row["temperatura"] > 35:
        puntaje += 2
    elif row["temperatura"] > 30:
        puntaje += 1
    
    # Gases
    if row["gas_ppm"] > 300:
        puntaje += 3
    elif row["gas_ppm"] > 150:
        puntaje += 2
    elif row["gas_ppm"] > 75:
        puntaje += 1
    
    # CO2
    if row["co2"] > 2000:
        puntaje += 2
    elif row["co2"] > 1200:
        puntaje += 1
    
    # Partículas
    if row["particulas_pm25"] > 120:
        puntaje += 2
    elif row["particulas_pm25"] > 60:
        puntaje += 1
    
    # Vibración
    if row["vibracion"] > 1.5:
        puntaje += 3
    elif row["vibracion"] > 0.8:
        puntaje += 2
    
    # Inclinación
    if row["inclinacion"] > 25:
        puntaje += 3
    elif row["inclinacion"] > 15:
        puntaje += 2
    elif row["inclinacion"] > 8:
        puntaje += 1
    
    # Obstáculos
    if row["distancia_obstaculo"] < 0.6:
        puntaje += 2
    elif row["distancia_obstaculo"] < 1.2:
        puntaje += 1
    
    # Batería
    if row["bateria"] < 25:
        puntaje += 2
    elif row["bateria"] < 45:
        puntaje += 1
    
    # Visibilidad
    if row["visibilidad"] < 25:
        puntaje += 2
    elif row["visibilidad"] < 50:
        puntaje += 1
    
    # Comunicación
    if row["senal_comunicacion"] < 35:
        puntaje += 2
    elif row["senal_comunicacion"] < 55:
        puntaje += 1
    
    if puntaje >= 9:
        return "Alto"
    elif puntaje >= 5:
        return "Medio"
    else:
        return "Bajo"

In [68]:
df["nivel_riesgo"] = df.apply(calcular_riesgo, axis=1)

df["nivel_riesgo"].value_counts()

nivel_riesgo
Alto     1035
Medio     837
Bajo      128
Name: count, dtype: int64

In [69]:
def recomendar_accion(row):
    if row["nivel_riesgo"] == "Bajo" and row["persona_detectada"] == 0:
        return "Avanzar"
    
    elif row["nivel_riesgo"] == "Bajo" and row["persona_detectada"] == 1:
        return "Enviar alerta y continuar exploración"
    
    elif row["nivel_riesgo"] == "Medio" and row["persona_detectada"] == 0:
        return "Avanzar con precaución"
    
    elif row["nivel_riesgo"] == "Medio" and row["persona_detectada"] == 1:
        return "Enviar alerta y avanzar con precaución"
    
    elif row["nivel_riesgo"] == "Alto" and row["persona_detectada"] == 0:
        return "Cambiar ruta o detenerse"
    
    elif row["nivel_riesgo"] == "Alto" and row["persona_detectada"] == 1:
        return "Enviar alerta y cambiar ruta"
    
    else:
        return "Revisar manualmente"

df["accion_recomendada"] = df.apply(recomendar_accion, axis=1)

df[["nivel_riesgo", "persona_detectada", "accion_recomendada"]].head()

,nivel_riesgo,persona_detectada,accion_recomendada
0,Alto,0,Cambiar ruta o detenerse
1,Alto,0,Cambiar ruta o detenerse
2,Alto,0,Cambiar ruta o detenerse
3,Alto,0,Cambiar ruta o detenerse
4,Medio,0,Avanzar con precaución


In [70]:
columnas_finales = [
    "zona",
    "temperatura",
    "humedad",
    "presion",
    "luz",
    "sonido_db",
    "co2",
    "particulas_pm25",
    "gas_tipo",
    "gas_ppm",
    "vibracion",
    "inclinacion",
    "distancia_obstaculo",
    "velocidad_robot",
    "senal_comunicacion",
    "bateria",
    "voltaje_bateria",
    "temperatura_bateria",
    "autonomia_estimada_min",
    "visibilidad",
    "persona_detectada",
    "confianza_persona",
    "nivel_riesgo",
    "accion_recomendada"
]

df = df[columnas_finales]

df.head()

,zona,temperatura,humedad,presion,luz,sonido_db,co2,particulas_pm25,gas_tipo,gas_ppm,...,senal_comunicacion,bateria,voltaje_bateria,temperatura_bateria,autonomia_estimada_min,visibilidad,persona_detectada,confianza_persona,nivel_riesgo,accion_recomendada
0,Punto crítico,23.742562,37.156921,951.246360,0.59255,54.215643,505.075257,136.682603,gas_desconocido,16.081237,...,51.8,36.395081,10.82,4,28.5,45.2,0,0.11,Alto,Cambiar ruta o detenerse
1,Punto crítico,22.381415,37.422549,900.000000,0.00000,70.790609,545.730161,168.990470,humo,447.921945,...,58.6,61.886486,12.10,24,45.9,28.5,0,0.05,Alto,Cambiar ruta o detenerse
2,Túnel estrecho,22.333412,19.738998,941.234135,0.00000,50.419725,2085.626242,20.303590,sin_gas,326.041341,...,33.5,51.792509,11.46,24,37.8,72.8,0,0.14,Alto,Cambiar ruta o detenerse
3,Túnel estrecho,21.993103,40.284164,938.066586,0.53330,18.377416,2098.968669,102.632636,amoniaco,368.769094,...,23.5,60.061447,11.91,4,47.4,54.9,0,0.12,Alto,Cambiar ruta o detenerse
4,Habitación bloqueada,24.145010,42.892223,944.360592,0.47405,34.733886,915.139906,49.429148,sin_gas,1.303732,...,35.1,33.434138,11.06,24,23.1,74.9,0,0.19,Medio,Avanzar con precaución


In [71]:
print("Dimensiones del dataset final:", df.shape)

print("\nValores nulos:")
print(df.isnull().sum())

print("\nDuplicados:")
print(df.duplicated().sum())

print("\nDistribución de nivel de riesgo:")
print(df["nivel_riesgo"].value_counts())

print("\nDistribución porcentual:")
print(df["nivel_riesgo"].value_counts(normalize=True) * 100)

Dimensiones del dataset final: (2000, 24)

Valores nulos:
zona                      0
temperatura               0
humedad                   0
presion                   0
luz                       0
sonido_db                 0
co2                       0
particulas_pm25           0
gas_tipo                  0
gas_ppm                   0
vibracion                 0
inclinacion               0
distancia_obstaculo       0
velocidad_robot           0
senal_comunicacion        0
bateria                   0
voltaje_bateria           0
temperatura_bateria       0
autonomia_estimada_min    0
visibilidad               0
persona_detectada         0
confianza_persona         0
nivel_riesgo              0
accion_recomendada        0
dtype: int64

Duplicados:
0

Distribución de nivel de riesgo:
nivel_riesgo
Alto     1035
Medio     837
Bajo      128
Name: count, dtype: int64

Distribución porcentual:
nivel_riesgo
Alto     51.75
Medio    41.85
Bajo      6.40
Name: proportion, dtype: float64


In [72]:
display(df.head(10))

,zona,temperatura,humedad,presion,luz,sonido_db,co2,particulas_pm25,gas_tipo,gas_ppm,...,senal_comunicacion,bateria,voltaje_bateria,temperatura_bateria,autonomia_estimada_min,visibilidad,persona_detectada,confianza_persona,nivel_riesgo,accion_recomendada
0,Punto crítico,23.742562,37.156921,951.246360,0.59255,54.215643,505.075257,136.682603,gas_desconocido,16.081237,...,51.8,36.395081,10.82,4,28.5,45.2,0,0.11,Alto,Cambiar ruta o detenerse
1,Punto crítico,22.381415,37.422549,900.000000,0.00000,70.790609,545.730161,168.990470,humo,447.921945,...,58.6,61.886486,12.10,24,45.9,28.5,0,0.05,Alto,Cambiar ruta o detenerse
2,Túnel estrecho,22.333412,19.738998,941.234135,0.00000,50.419725,2085.626242,20.303590,sin_gas,326.041341,...,33.5,51.792509,11.46,24,37.8,72.8,0,0.14,Alto,Cambiar ruta o detenerse
3,Túnel estrecho,21.993103,40.284164,938.066586,0.53330,18.377416,2098.968669,102.632636,amoniaco,368.769094,...,23.5,60.061447,11.91,4,47.4,54.9,0,0.12,Alto,Cambiar ruta o detenerse
4,Habitación bloqueada,24.145010,42.892223,944.360592,0.47405,34.733886,915.139906,49.429148,sin_gas,1.303732,...,35.1,33.434138,11.06,24,23.1,74.9,0,0.19,Medio,Avanzar con precaución
5,Pasillo A,22.183517,40.978955,951.287138,0.00000,15.188540,1691.979226,29.335347,amoniaco,314.119717,...,74.3,70.361328,12.00,4,55.7,63.9,0,0.11,Medio,Avanzar con precaución
6,Pasillo A,18.488869,32.793439,900.000000,0.00000,62.342806,1330.678158,43.321540,sin_gas,294.243082,...,26.6,60.061447,11.83,4,47.4,70.9,0,0.14,Alto,Cambiar ruta o detenerse
7,Pasillo A,23.008155,39.166320,938.372606,0.00000,25.928363,1728.661687,106.108537,metano,268.602778,...,64.4,60.061447,11.58,24,44.4,41.7,0,0.36,Alto,Cambiar ruta o detenerse
8,Sector norte,21.939792,24.852293,900.000000,0.00000,43.212154,2212.046497,16.398538,amoniaco,392.933428,...,31.7,56.820814,11.84,24,41.9,60.2,0,0.14,Alto,Cambiar ruta o detenerse
9,Punto crítico,24.989807,34.484990,955.067005,1.71520,70.218749,567.698492,80.988254,sin_gas,277.935548,...,35.7,60.061447,11.73,4,47.4,61.2,1,0.86,Medio,Enviar alerta y avanzar con precaución


In [73]:
output_path = "../data/processed/rescuetwin_dataset.csv"

df.to_csv(output_path, index=False)

print("Dataset final guardado correctamente en:")
print(output_path)

Dataset final guardado correctamente en:
../data/processed/rescuetwin_dataset.csv


In [74]:
df_verificacion = pd.read_csv("../data/processed/rescuetwin_dataset.csv")

print("Dimensiones:", df_verificacion.shape)
display(df_verificacion.head())

Dimensiones: (2000, 24)


,zona,temperatura,humedad,presion,luz,sonido_db,co2,particulas_pm25,gas_tipo,gas_ppm,...,senal_comunicacion,bateria,voltaje_bateria,temperatura_bateria,autonomia_estimada_min,visibilidad,persona_detectada,confianza_persona,nivel_riesgo,accion_recomendada
0,Punto crítico,23.742562,37.156921,951.246360,0.59255,54.215643,505.075257,136.682603,gas_desconocido,16.081237,...,51.8,36.395081,10.82,4,28.5,45.2,0,0.11,Alto,Cambiar ruta o detenerse
1,Punto crítico,22.381415,37.422549,900.000000,0.00000,70.790609,545.730161,168.990470,humo,447.921945,...,58.6,61.886486,12.10,24,45.9,28.5,0,0.05,Alto,Cambiar ruta o detenerse
2,Túnel estrecho,22.333412,19.738998,941.234135,0.00000,50.419725,2085.626242,20.303590,sin_gas,326.041341,...,33.5,51.792509,11.46,24,37.8,72.8,0,0.14,Alto,Cambiar ruta o detenerse
3,Túnel estrecho,21.993103,40.284164,938.066586,0.53330,18.377416,2098.968669,102.632636,amoniaco,368.769094,...,23.5,60.061447,11.91,4,47.4,54.9,0,0.12,Alto,Cambiar ruta o detenerse
4,Habitación bloqueada,24.145010,42.892223,944.360592,0.47405,34.733886,915.139906,49.429148,sin_gas,1.303732,...,35.1,33.434138,11.06,24,23.1,74.9,0,0.19,Medio,Avanzar con precaución


## Fuentes de datos utilizadas

| Fuente | Tipo de dato | Uso dentro del proyecto |
|---|---|---|
| Indoor Environmental Dataset | Sensores ambientales | Temperatura, humedad, presión, luz, sonido, CO2 y partículas |
| UCI Gas Sensor Dataset | Sensores químicos | Tipo de gas y concentración estimada |
| NASA Battery Dataset Cleaned | Datos de batería | Batería restante, voltaje, temperatura de batería y autonomía estimada |
| SARD Search and Rescue Dataset | Imágenes de rescate | Referencia para simular detección de personas atrapadas |

## Justificación

No existe un único dataset público que represente completamente un robot cuadrúpedo operando dentro de una zona de derrumbe. Por este motivo, se decidió construir un dataset integrado a partir de varias fuentes públicas y variables simuladas coherentes con el funcionamiento de un robot de rescate.

El dataset final representa lecturas de sensores ambientales, químicos, estructurales y operativos del robot. A partir de estas variables se genera una clasificación de riesgo operativo: bajo, medio o alto.